# Data Understanding

## Import Libraries and Set Global Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

## Define Data Sources

### Load Datasets

In [ ]:
# Define Dataset Paths
datasets = {
    "uk_dataset": glob.glob("../data/raw/100,000 UK Used Car Data set/*.csv"),
    "andrei_dataset": "../data/raw/Used Cars Dataset Andrei Novikov/cars.csv"
}

# Assign dataframes variable
dfs = {}

# Iterate through datasets
for name, path in datasets.items():
    try:
        # Check if the path contains multiple files or is a single file
        if isinstance(path, list):
            # Read all files in the list, extract make, and concatenate them
            df_list = []
            for f in path:
                temp_df = pd.read_csv(f)
                
                # Extract filename without extension to use as the 'make'
                if name == "uk_dataset":
                    make_name = os.path.basename(f).replace('.csv', '').strip().lower()
                    
                    # Handle inconsistent make names from model named files (UK Dataset)
                    if make_name in ['merc', 'cclass']:
                        make_name = 'mercedes'
                    elif make_name == 'focus':
                        make_name = 'ford'
                        
                    # Add make names into the temporary dataframe
                    temp_df['make'] = make_name.capitalize()
                
                # Append the temporary dataframe with the updated make names to the dataframe list (UK Dataset)
                df_list.append(temp_df)
                
            # Append make names into a new feature in the dataset
            dfs[name] = pd.concat(df_list, ignore_index=True)
            
        else:
            # It's a single file path
            dfs[name] = pd.read_csv(path)
            
        # Output Dataset name and shape
        print(f"Loaded {name}: {dfs[name].shape[0]} rows, {dfs[name].shape[1]} columns.")
    except Exception as e:
        # Output Error if dataset not found
        print(f"Error loading {name}: {e}")

### Output Sample from All Datasets

In [ ]:
# Iterate through datasets
for i in dfs.keys():
    # Output Dataset Name
    print(f'\n{'='*20} {i} {'='*20}')
    # Output Samples of the dataset
    print(dfs[i].sample(5))

## Data Source Documentation

In [ ]:
# Document data source details for each dataset
data_source_report = {
    
    # 100,000 UK Dataset
    'uk_dataset': {
        'name': '100,000 UK Used Car Data set'
        'source': 'https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes',
        'acquisition_method': 'CSV download through kaggle.com (multiple files per car make)',
        'date_acquired': '2026',
        'issues_encountered': ['Multiple CSV files need merging', 'Inconsistent make names (cclass, merc, focus)']
    },
    
    # Andrei Novikov Dataset
    'andrei_dataset': {
        'name': 'Used Cars Dataset by Andrei Noviko'
        'source': 'https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset',
        'acquisition_method': 'CSV download through kaggle.com',
        'date_acquired': '2026',
        'issues_encountered': ['Missing mpg and engineSize columns', 'US-based data (different market)']
    }
}

# Print the documented data to console/output
for name, report in data_source_report.items():
    print(f"\n{'='*20} {name} {'='*20}")
    for key, value in report.items():
        print(f"  {key}: {value}")

## Describe Data

### Dataset Information

In [ ]:
# Iterate through datasets
for name, df in dfs.items():
    # Output dataset name
    print(f"\n{'='*20} {name} {'='*20}")
    # Output dataset information
    print(df.info())

### Numerical Statistics

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Output dataset name header
    print(f"\n--- Numerical Statistics ({name}) ---")
    # Output dataset description
    print(df.describe()) 

### Categorical Statistics

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Categorical Overview
    if not df.select_dtypes(include='object').empty:

        # Output dataset name header
        print(f"\n--- Categorical Statistics ({name}) ---")
        # Output categorical feature Statistics
        print(df.describe(include='object'))

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Output dataset name header
    print(f"\n--- {name} Object Count ---")

    # Identify object-type columns in the current dataframe
    object_cols = df.select_dtypes(include='object').columns
    
    # Check if object columns exist before iterating
    if len(object_cols) > 0:
        # Iterate through identified categorical columns
        for col in object_cols:
            # Output column name header
            print(f"\n--- {col} ---")
            
            # Output top 20 value counts for the specific column
            print(df[col].value_counts().head(20)) 
    else:
        # Output message if no object columns are present in the dataset
        print(f"No object-type columns found in {name}.")

## Explore Data

### UK Dataset

#### Univariate Analysis

##### Price and Milage Distribution

In [ ]:
# Initialize subplots for price and mileage
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set title for the combined plots
fig.suptitle('Price and Mileage Distributions (UK Dataset)', fontsize=16)

# Generate histogram and KDE for price
sns.histplot(dfs['uk_dataset']['price'], bins=50, kde=True, ax=axes[0], color='skyblue')
    
# Set individual plot title and axis labels for price
axes[0].set_title('Distribution of Price (UK Dataset)')
axes[0].set_xlabel('Price (GBP)')

# Generate histogram and KDE for mileage
sns.histplot(dfs['uk_dataset']['mileage'], bins=50, kde=True, ax=axes[1], color='lightgreen')
    
# Set individual plot title and axis labels for mileage
axes[1].set_title('Distribution of Mileage (UK Dataset)')
axes[1].set_xlabel('Mileage')

# Adjust layout to prevent overlapping and display the plot
plt.tight_layout()

# Show histogram plots
plt.show()

##### Transmission and Fuel Type Count

In [ ]:
# Initialize subplots for transmission and fuel type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set title for the categorical analysis plots
fig.suptitle('Transmission and Fuel Type Analysis (UK Dataset)', fontsize=16)

# Generate count plot for transmission types
sns.countplot(data=dfs['uk_dataset'], x='transmission', ax=axes[0], palette='Set2')
    
# Set individual plot title for transmission
axes[0].set_title('Count of Vehicles by Transmission (UK Dataset)')
    
# Annotate bars with frequency labels for better readability
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Generate count plot for fuel types
sns.countplot(data=dfs['uk_dataset'], x='fuelType', ax=axes[1], palette='Set2')
    
# Set individual plot title and apply log scale for better visualization of rare categories
axes[1].set_title('Count of Vehicles by Fuel Type (UK Dataset)')
axes[1].set_ylabel('Count')
    
# Annotate bars with frequency labels
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

# Adjust layout to optimize spacing between subplots
plt.tight_layout()
# Show countplots
plt.show()

##### Make and Model Distribution

In [ ]:
# Initialize subplots for make and model variety analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Set title for the make-level overview
fig.suptitle('Make and Model Analysis (UK Dataset)', fontsize=16)

# Determine the order of make by frequency
make_order = dfs['uk_dataset']['make'].value_counts().index
    
# Generate horizontal count plot for make frequency
sns.countplot(data=dfs['uk_dataset'], y='make', order=make_order, ax=axes[0], palette='viridis')
    
# Set individual plot title and axis labels for make counts
axes[0].set_title('Total Count of Vehicles by Make (UK Dataset)')
axes[0].set_xlabel('Number of Vehicles')
axes[0].set_ylabel('Make')
    
# Annotate bars with total vehicle counts per make
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Calculate the number of unique models available for each make
unique_models = dfs['uk_dataset'].groupby('make')['model'].nunique().sort_values(ascending=False)
    
# Generate horizontal bar plot
sns.barplot(x=unique_models.values, y=unique_models.index, ax=axes[1], palette='magma')
    
# Set individual plot title and labels for unique model counts
axes[1].set_title('Variety of Inventory: Number of Unique Models per Make (UK Dataset)')
axes[1].set_xlabel('Count of Unique Models')
axes[1].set_ylabel('Make')
    
# Annotate bars with the specific count of unique models
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

# Optimize layout for readability and display the combined figure
plt.tight_layout()
# Show barplots
plt.show()

#### Bivariate/Multivariate Analysis

##### Price vs Milage (Best Fit Line)

In [ ]:
# Set figure size
plt.figure(figsize=(10, 6))

# Generate scatter plot 
sns.scatterplot(
    data=dfs['uk_dataset'].sample(min(10000, len(dfs['uk_dataset']))), 
    x='mileage', y='price', alpha=0.3,  s=15, color='skyblue'
)

# Add a linear regression line to show the general trend of depreciation
sns.regplot(
    data=dfs['uk_dataset'], 
    x='mileage', 
    y='price', 
    scatter=False,
    color='red', 
    line_kws={"linewidth": 2}
)

# Set x and Y limit
plt.ylim(bottom=0, top=60000)
plt.xlim(0, 150000) 
    
# Add descriptive labels and title
plt.title('Price vs. Mileage (UK Dataset)', fontsize=14)
plt.xlabel('Mileage')
plt.ylabel('Price (GBP)')

# Show Scatterplot
plt.show()

##### Price vs Milage Hexbin

In [ ]:
# Create a hex mask for price < 40000 and milage < 80000
uk_hex_mask = (dfs['uk_dataset']['price'] < 40000) & (dfs['uk_dataset']['mileage'] < 80000)

# Extract price and milage data
uk_hex_data = dfs['uk_dataset'][uk_hex_mask]

# Generate hexbin plot using jointplot
hb = sns.jointplot(data=uk_hex_data, x='mileage', y='price', kind='hex', cmap='YlOrRd', gridsize=40, height=7)

# Set axis labels
hb.set_axis_labels('Mileage', 'Price (GBP)')

# Set graph title
plt.suptitle("Density of Price vs Mileage (UK Dataset)", y=1.02, fontsize=14)

# Show hexbin
plt.show()

##### Price by Transmission

In [ ]:
# Set figure size
plt.figure(figsize=(10, 6))

# Generate boxplots
sns.boxplot(data=dfs['uk_dataset'], x='transmission', y='price', palette='Set2', showfliers=False)

# Add descriptive labels and title
plt.title('Price by Transmission (UK Dataset)', fontsize=14)
plt.xlabel('Transmission')
plt.ylabel('Price (GBP)')

# Show boxplots
plt.show()

### Andrei Dataset

#### Univariate Analysis

##### Price and Mileage Distribution

In [ ]:
# Create a mask to handle outliers price < 100000 and milage < 300000
vis_mask = (dfs['andrei_dataset']['price'] < 100000) & (dfs['andrei_dataset']['mileage'] < 300000)

# Apply the mask to create a filtered dataframe
df_andrei = dfs['andrei_dataset'][vis_mask]

# Initialize subplots for price and mileage distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set the main title for the combined plots
fig.suptitle('Price and Mileage (Andrei Dataset)', fontsize=16)

# Generate histogram and KDE for price distribution
sns.histplot(df_andrei['price'], bins=50, kde=True, ax=axes[0], color='salmon')

# Set individual plot title and axis labels for price
axes[0].set_title('Distribution of Price (< $100k)')
axes[0].set_xlabel('Price (USD)')

# Generate histogram and KDE for mileage distribution
sns.histplot(df_andrei['mileage'], bins=50, kde=True, ax=axes[1], color='gold')

# Set individual plot title and axis labels for mileage
axes[1].set_title('Distribution of Mileage (< 300k)')
axes[1].set_xlabel('Mileage')

# Adjust layout to prevent overlapping
plt.tight_layout()

# Show distribution plots
plt.show()

##### Transmission and Fuel Type Count

In [ ]:
# Initialize subplots for transmission and fuel type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set the main title for the categorical plots
fig.suptitle('Andrei Dataset: Top 5 Transmission and Fuel Types', fontsize=16)

# Identify the top 5 most frequent transmission types
top_trans = df_andrei['transmission'].value_counts().nlargest(5).index

# Generate horizontal count plot for the top 5 transmission types
sns.countplot(
    data=df_andrei[df_andrei['transmission'].isin(top_trans)], 
    y='transmission', 
    order=top_trans, 
    ax=axes[0], 
    palette='Set2'
)

# Set individual plot title for transmission
axes[0].set_title('Top 5 Transmissions')

# Annotate bars with frequency labels for readability
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Identify the top 5 most frequent fuel types
top_fuel = df_andrei['fuel_type'].value_counts().nlargest(5).index

# Generate horizontal count plot for the top 5 fuel types
sns.countplot(
    data=df_andrei[df_andrei['fuel_type'].isin(top_fuel)], 
    y='fuel_type', 
    order=top_fuel, 
    ax=axes[1], 
    palette='Set2'
)

# Set individual plot title and apply log scale to the x-axis for better visualization
axes[1].set_title('Top 5 Fuel Types (Log Scale)')
axes[1].set_xscale('log')
axes[1].set_xlabel('Count (Log Scale)')

# Annotate bars with frequency labels
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

# Adjust layout to prevent overlapping
plt.tight_layout()

# Show countplots
plt.show()

##### Make and Model Distribution

In [ ]:
# Initialize subplots for manufacturer and model variety analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 16))

# Set the main title for the manufacturer-level overview
fig.suptitle('Top 20 Manufacturer and Model Analysis (Andrei Dataset)', fontsize=16)

# Identify the top 20 manufacturers by total vehicle count
make_order = df_andrei['manufacturer'].value_counts().nlargest(20).index

# Generate horizontal count plot for manufacturer frequency
sns.countplot(
    data=df_andrei[df_andrei['manufacturer'].isin(make_order)], 
    y='manufacturer', 
    order=make_order, 
    ax=axes[0], 
    palette='viridis'
)

# Set individual plot title and axis labels for manufacturer counts
axes[0].set_title('Top 20 Manufacturers by Total Count')
axes[0].set_xlabel('Number of Vehicles')

# Annotate bars with total vehicle counts per manufacturer
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Calculate the number of unique models available for each of the top 20 manufacturers
unique_models = df_andrei.groupby('manufacturer')['model'].nunique().sort_values(ascending=False).head(20)

# Generate horizontal bar plot for unique model counts
sns.barplot(x=unique_models.values, y=unique_models.index, ax=axes[1], palette='magma')

# Set individual plot title and labels for unique model counts
axes[1].set_title('Top 20 Manufacturers by Variety (Unique Models)')
axes[1].set_xlabel('Count of Unique Models')

# Adjust layout to prevent overlapping
plt.tight_layout()

# Show barplots
plt.show()

#### Bivariate/Multivariate Analysis

##### Price vs Milage

In [ ]:
# Create a mask to filter out outliers
vis_mask = (dfs['andrei_dataset']['price'] < 100000) & (dfs['andrei_dataset']['mileage'] < 300000)

# Apply the mask to create a filtered dataframe
df_andrei = dfs['andrei_dataset'][vis_mask]

# Set figure size for the scatter plot
plt.figure(figsize=(10, 6))

# Generate scatter plot
sns.scatterplot(
    data=df_andrei.sample(min(10000, len(df_andrei))), 
    x='mileage', 
    y='price', 
    alpha=0.3, 
    s=15, 
    color='purple'
)

# Add a linear regression line to show the general trend of depreciation
sns.regplot(
    data=df_andrei, 
    x='mileage', 
    y='price', 
    scatter=False, 
    color='red', 
    line_kws={"linewidth": 2}
)

# Set the lower limit of the y-axis to 0 to prevent negative price values
plt.ylim(bottom=0)

# Add descriptive labels and title
plt.title(': Price vs. Mileage (Andrei Dataset)', fontsize=14)
plt.xlabel('Mileage')
plt.ylabel('Price (USD)')

# Show scatterplot
plt.show()

##### Price to Accident Ratio

In [ ]:
# Set figure size for the boxplot
plt.figure(figsize=(10, 6))

# Generate boxplots to compare price distributions based on accident history
sns.boxplot(
    data=df_andrei, 
    x='accidents_or_damage', 
    y='price', 
    palette='Set1', 
    showfliers=False
)

# Add descriptive labels and title
plt.title('Price by Accident History (Andrei Dataset)', fontsize=14)
plt.xlabel('Accidents or Damage (0.0 = No, 1.0 = Yes)')
plt.ylabel('Price (USD)')

# Show boxplots
plt.show()

##### Correlation Matrix

In [ ]:
# Define features to drop due to irrelevance or high missing values
cols_to_drop = ['one_owner', 'driver_reviews_num', 'seller_rating']

# Drop specified features from the dataset
dfs['andrei_dataset'].drop(columns=cols_to_drop, errors='ignore', inplace=True)

# Extract numeric engine size from the string format using regular expression
dfs['andrei_dataset']['engine_size_numeric'] = dfs['andrei_dataset']['engine'].astype(str).str.extract(r'(\d+\.\d+)').astype(float)

# Define function to calculate the average mpg from a string
def parse_mpg(x):
    # Return NaN if the value is missing
    if pd.isna(x): 
        return np.nan
    try:
        # Split the string by hyphen and calculate the mean of the numeric parts
        parts = str(x).split('-')
        return np.mean([float(p) for p in parts])
    except:
        # Return NaN if parsing fails
        return np.nan

# Apply the parse_mpg function to create a new numeric mpg feature
dfs['andrei_dataset']['mpg_numeric'] = dfs['andrei_dataset']['mpg'].apply(parse_mpg)

# Re-apply the visibility mask to update the visualization dataframe with new numeric columns
df_andrei = dfs['andrei_dataset'][(dfs['andrei_dataset']['price'] < 100000) & (dfs['andrei_dataset']['mileage'] < 300000)]

# Set figure size for the correlation heatmap
plt.figure(figsize=(10, 8))

# Select only the numerical columns for correlation analysis
num_cols_andrei = df_andrei.select_dtypes(include=[np.number]).columns

# Calculate the Pearson correlation coefficients for the numerical features
corr_andrei = df_andrei[num_cols_andrei].corr()

# Generate heatmap with annotations, custom color map, and fixed value ranges (-1 to 1)
sns.heatmap(corr_andrei, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)

# Set the title for the correlation matrix
plt.title('Correlation Matrix (Andrei Dataset)', fontsize=14)

# Show the heatmap
plt.show()

##### Price vs Milage Hexbin

In [ ]:
# Create a hex mask for price < 60000 and milage < 200000
andrei_hex_mask = (df_andrei['price'] < 60000) & (df_andrei['mileage'] < 200000)

# Extract price and milage data
andrei_hex_data = df_andrei[andrei_hex_mask]

# Generate hexbin plot using jointplot
g = sns.jointplot(
    data=andrei_hex_data, 
    x='mileage', 
    y='price', 
    kind='hex', 
    cmap='YlOrRd', 
    gridsize=40, 
    height=7
)

# Set axis labels for the jointplot
g.set_axis_labels('Mileage', 'Price (USD)')

# Set title for hexbin plot
plt.suptitle("Density of Price vs Mileage (Andrei Dataset)", y=1.02, fontsize=14)

# Show hexbin plot
plt.show()

## Verify Data Quality

In [ ]:
for name, df in dfs.items():
    # Check for feature missing values
    missing = df.isnull().sum()
    # Check for the dataset's total missing value
    total_missing = missing.sum()

    # Check for duplicate records
    duplicates = df.duplicated.sum()

    # Check for dataset name to set max year based on the date of the dataset
    match name:
        case "uk_dataset":
            max_year = 2020
        case "andrei_dataset":
            max_year = 2023
        case _:
            max_year = 2026

    # Check if year exists as a feature in the dataset
    if 'year' in df.columns:
        # Extract future car error based on a boolean mask
        future_cars = df[df['year'] > max_year].shape[0]

    # Check if engine size exists as a feature in the dataset
    if 'engineSize' in df.columns:
        engine_check = df[df['engineSize'] < 0.6 | df['engineSize'] > 8.0]

    # Print summary for dataset
    print(f'\n{'='*20} {name} {'='*20}')
    print(f"Total Rows: {len(df):,}")
    print(f"Duplicate Rows: {dupes:,} ({(dupes/len(df))*100:.2f}%)")
    print(f"Future Year Entries (>{max_year}): {future_cars}")
    if engine_check > 0:
        print(f"Zero Engine Size Entries: {engine_check}")

    for col in df.columns:
        # Gather feature metrics
        dtype_info = df[col].dtype
        missing_count = missing[col]
        
        # Calculate missing percentage per feature
        if total_rows > 0:
            missing_percentage = (missing_count / total_rows) * 100
        else:
            missing_percentage = 0.0
        
        # Print feature metrics
        print(f"Column: {col:<15} | "
            f"Type: {str(dtype_info):<10} | "
            f"Missing: {missing_count:<6} | "
            f"Percentage: {missing_percentage:.2f}%")